# KGP-One Independent Entity Extraction
This notebook loads the chunks generated from the previous step (`chunks_output.json`) and uses GLiNER to extract NLP entities (Algorithms, Concepts, Metrics, etc.).\n\n**It is 100% independent and contains all required classes inline.**

In [ ]:
# !pip install -q gliner


In [ ]:
from abc import ABC, abstractmethod
from typing import Dict, Any, List

class BaseEntityExtractor(ABC):
    @abstractmethod
    async def extract(self, chunks: List[str], context: Dict[str, Any]) -> Dict[str, Any]:
        pass


In [ ]:
import logging
import os
from typing import Any

logger = logging.getLogger("model_factory")

class NLPModelFactory:
    _gliner_model = None

    @classmethod
    def get_gliner(cls) -> Any:
        if cls._gliner_model is None:
            model_name = os.environ.get("GLINER_MODEL_NAME", "urchade/gliner_medium-v2.1")
            print(f"Loading GLiNER model: {model_name}...")
            from gliner import GLiNER
            cls._gliner_model = GLiNER.from_pretrained(model_name)
        return cls._gliner_model


In [ ]:
import warnings
warnings.filterwarnings("ignore", message="The `resume_download` argument is deprecated")

class GLiNERExtractor(BaseEntityExtractor):
    def __init__(self, labels: List[str] = None):
        self.labels = labels or [
            "Concept", "Algorithm", "Dataset", "Metric", 
            "Author", "Tool", "Task", "Methodology"
        ]
        
    async def extract(self, chunks: List[str], context: Dict[str, Any]) -> Dict[str, Any]:
        try:
            model = NLPModelFactory.get_gliner()
        except Exception as e:
            print(f"Failed to load GLiNER: {e}")
            return {"entities": []}

        extracted_entities = []
        for chunk in chunks:
            try:
                entities = model.predict_entities(chunk, self.labels)
                for ent in entities:
                    extracted_entities.append({
                        "text": ent["text"],
                        "label": ent["label"],
                        "score": ent.get("score", 1.0)
                    })
            except Exception as e:
                print(f"Error during GLiNER extraction on chunk: {e}")
                
        # Deduplicate naive implementation
        seen = set()
        unique_entities = []
        for ent in extracted_entities:
            key = (ent["text"].lower(), ent["label"])
            if key not in seen:
                seen.add(key)
                unique_entities.append(ent)
                
        return {"entities": unique_entities}


In [ ]:
import json
import asyncio

async def run_extraction():
    # 1. Load the chunks output from the previous notebook
    try:
        with open("chunks_output.json", "r", encoding="utf-8") as f:
            chunks = json.load(f)
    except FileNotFoundError:
        print("chunks_output.json not found! Using some sample text instead.")
        chunks = [
            {"content": "The Transformer algorithm is a deep learning architecture developed by Google in 2017. It relies entirely on the self-attention mechanism, dispensing with recurrences and convolutions."},
            {"content": "We evaluated our approach on the WMT 2014 English-to-German translation task, achieving a BLEU score of 28.4."}
        ]
        
    chunk_texts = [c["content"] for c in chunks[:10]] # Limit to 10 chunks for speed
    print(f"Extracting entities from {len(chunk_texts)} chunks...")
    
    # 2. Run GLiNER Extraction
    extractor = GLiNERExtractor()
    result = await extractor.extract(chunk_texts, context={})
    
    entities = result.get("entities", [])
    print(f"\n--- EXTRACTED {len(entities)} UNIQUE ENTITIES ---")
    
    # 3. Save nicely to JSON
    with open("entities_output.json", "w", encoding="utf-8") as f:
        json.dump(entities, f, indent=4, ensure_ascii=False)
        
    print("Saved output to 'entities_output.json'")
    
    # Print preview
    for e in entities[:10]:
        print(f"[{e['label']}] {e['text']} (Confidence: {e['score']:.2f})")

# Run it!
await run_extraction()
